# Exercise 4: storyboard → clips → measure consistency

## Tasks

1. **Plan:** let the API draft a short storyboard.
2. **Generate:** create one clip in Flow with the character sheet as reference.
3. **Check:** compare the character in the clip with the sheet.
4. **Extension:** more scenes, joined into one film with ffmpeg.

Required: 1 video. No Flow video available? Use Ueli's clip from the course data.

## Setup

Select the **Python 3.11** kernel. The API key comes from your `OPENAI_API_KEY` Codespaces secret.

In [ ]:
# Standard Python libraries.
import base64
import json
import mimetypes
import os
from pathlib import Path
from typing import Literal

# Libraries installed from requirements.txt.
import cv2
import pandas as pd
from openai import OpenAI
from PIL import Image, ImageOps
from pydantic import BaseModel

# Work inside this exercise folder, wherever Jupyter starts.
if not Path("data").exists():  # we are inside an exercise folder
    os.chdir("..")  # go to the main folder
os.chdir("04_Storyboard_Consistency")

# Connect to OpenAI. The key is read from OPENAI_API_KEY; busy requests are retried.
client = OpenAI(max_retries=5, timeout=60)
MODEL = "gpt-6-luna"

# Folder with the course images and videos.
DATA = Path("..") / "data"

## Helper functions for the API calls

- `image_to_input(path)` turns an image file into text (base64), because the API request is sent as text.
- `ask(prompt, answer_format, images)` sends the prompt and the images, writes a log line to `api_log.jsonl` and returns the answer as a Python dict.

In [ ]:
# Function 1: turn an image file into an API input.
def image_to_input(path):
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }


# Function 2: send a prompt and images, get the answer as a dict.
def ask(prompt, answer_format, images):
    # 1. Put the text and the images into one message.
    content = [{"type": "input_text", "text": prompt}]
    for path in images:
        content.append(image_to_input(path))

    # 2. Send the message. The answer must follow answer_format.
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        text_format=answer_format,
        reasoning={"effort": "none"},
        max_output_tokens=500,
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Stop with a clear message if there is no valid answer.
    if response.output_parsed is None:
        raise ValueError(f"No valid answer for {images}. Look at api_log.jsonl.")

    # 5. Return the answer as a Python dict.
    return response.output_parsed.model_dump()

## 1. Let the API draft a short storyboard

Plan the scenes first, prompt each scene second. The character descriptions stay **exactly the same** in every prompt.

In [ ]:
# Fixed descriptions, used in every prompt.
CHARACTER = "Ueli, red beanie, olive jacket"
DOG = "Bruno, Bernese, red collar"


# One scene of the storyboard.
class Scene(BaseModel):
    id: int
    shot: str
    flow_prompt: str


# The whole storyboard: a list of scenes.
class Storyboard(BaseModel):
    scenes: list[Scene]

In [ ]:
# The instruction for the model.
PROMPT = (
    "Write a storyboard with four short scenes for Google Flow, numbered 1 to 4: "
    "1 a wide mountain lake without characters, 2 a drifting boat, "
    f"3 {CHARACTER} together with the dog {DOG}, 4 sunset. "
    "For shot, give the shot size, e.g. wide or close-up. "
    "Each flow_prompt describes camera, action, light and sound."
)

# Ask for the plan (no images needed) and show it as a table.
plan = ask(PROMPT, Storyboard, [])
pd.DataFrame(plan["scenes"])

In [ ]:
# The Flow prompt for scene 3, with the fixed descriptions in front.
scene_3 = plan["scenes"][2]["flow_prompt"]
print(f"{CHARACTER}. {DOG}. {scene_3}")

## 2. Create one clip in Flow with the character sheet

Use **Veo 3.1 Lite**, 8 seconds and **Ingredients to Video**. Add `ueli_sheet.jpg` as ingredient and paste the prompt above. Upload the clip as `my_scene.mp4` next to this notebook.

In [ ]:
# Show the character sheet.
sheet = DATA / "reference" / "ueli_sheet.jpg"
display(ImageOps.contain(Image.open(sheet), (400, 300)))

# Use your own clip, or the course clip if there is none.
video_path = Path("my_scene.mp4")
if not video_path.exists():
    video_path = DATA / "videos" / "ueli_scene3_talk.mp4"
print("Using", video_path.name)

## 3. Compare the character in the clip with the sheet

One frame per second is compared with the character sheet.

In [ ]:
# Create the frames folder and open the video.
Path("frames").mkdir(exist_ok=True)
video = cv2.VideoCapture(str(video_path))

# Save one frame per second as a JPG file.
paths = []
for t in range(8):  # Flow clips are 8 seconds long
    video.set(cv2.CAP_PROP_POS_MSEC, t * 1000)  # jump to second t
    ok, frame = video.read()
    if not ok:  # the video ended earlier
        break
    path = Path("frames") / f"frame_{t}s.jpg"
    cv2.imwrite(str(path), frame)
    paths.append(path)

# Close the video file.
video.release()
print(len(paths), "frames saved in frames/")

In [ ]:
# The three allowed ratings.
Rating = Literal["match", "differs", "not visible"]


# The answer format: one rating per feature.
class Consistency(BaseModel):
    beanie: Rating
    jacket: Rating
    dog: Rating


# The instruction: compare image 2 with the reference image 1.
PROMPT = (
    "Image 1 is the reference. Rate the beanie, jacket and dog in image 2. "
    "Use match for a visible match, differs for a visible change, "
    "and not visible when unclear or out of frame."
)

In [ ]:
# Compare every frame with the sheet (reference first).
rows = []
for path in paths:
    row = ask(PROMPT, Consistency, [sheet, path])
    row["frame"] = path.name
    rows.append(row)

# One row per frame.
ratings = pd.DataFrame(rows)
display(ratings)

# Share of "match" per feature.
(ratings[["beanie", "jacket", "dog"]] == "match").mean()

## Extension: join more scenes into one film

Create more scenes in Flow, upload them next to this notebook and list them in story order. `-c copy` joins the clips without re-encoding – they need the same format, resolution and frame rate.

In [ ]:
# Your clips, in story order.
clips = ["scene_1.mp4", "scene_2.mp4", "scene_3.mp4", "scene_4.mp4"]

# ffmpeg reads the list of clips from a text file.
with open("scenes.txt", "w") as file:
    for clip in clips:
        file.write(f"file '{clip}'\n")

# Join the clips into one film.
!ffmpeg -y -f concat -i scenes.txt -c copy film.mp4

**Check yourself:** If every frame were generated as a separate image, the video would flicker (details jump from frame to frame). Why – and how do video models avoid it?

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')